# SpendDNA – Week 2 Industry-Graded Minor Project

## Name:
Srishti B S

## Roll Number:
17999

## Batch:
B.Tech CSE (Data Science)

## Date:
07 August 2026

### Objective

Analyze six months of banking transactions, clean messy financial data, extract vendors, categorize spending, detect anomalies, identify spending archetypes, and generate a professional spending analytics report using only Python, Pandas, and NumPy.


In [2]:
# Import Pandas for tabular data handling and NumPy for numerical calculations
import pandas as pd
import numpy as np

In [5]:
#Loading the provided CSV dataset into Pandas.

df = pd.read_csv("DADS MP2 Dataset.csv")

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

Dataset loaded successfully.
Dataset shape: (1328, 8)


In [6]:
#  Initial inspection of the dataset.

df.head()

,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [7]:
print("Column names:")
print(df.columns.tolist())

Column names:
['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref']


In [8]:
# Basic structural inspection.

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nData types:")
print(df.dtypes)

Number of rows: 1328
Number of columns: 8

Data types:
Date            object
Time            object
Description     object
Type            object
Amount          object
Balance        float64
Mode            object
Ref             object
dtype: object


In [9]:
print("Missing values in each column:")
print(df.isnull().sum())

Missing values in each column:
Date           0
Time           0
Description    0
Type           0
Amount         0
Balance        0
Mode           0
Ref            0
dtype: int64


In [11]:
print("Number of exact duplicate rows:")
print(df.duplicated().sum())


Number of exact duplicate rows:
18


In [13]:
print("Transaction type values:")
print(df["Type"].value_counts(dropna=False))

Transaction type values:
Type
DR       670
Debit    652
CR         6
Name: count, dtype: int64


FEATURE 1: TRANSACTION PARSER

In [14]:
# Creating a separate DataFrame so that the original dataset remains unchanged.

df_clean = df.copy()

print("Clean DataFrame created.")

Clean DataFrame created.


In [18]:
# AI-assisted: Parsing mixed date formats using Pandas datetime functionality.
# The dataset contains multiple valid date formats, so format="mixed"
# allows Pandas to evaluate each value individually.

df_clean["date"] = pd.to_datetime(
    df_clean["Date"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)

print(df_clean[["Date", "date"]].head(10))

          Date       date
0   2024-01-01 2024-01-01
1    01-Jan-24 2024-01-01
2    01-Jan-24 2024-01-01
3   2024-01-01 2024-01-01
4  01 Jan 2024 2024-01-01
5   2024-01-01 2024-01-01
6   2024-01-01 2024-01-01
7    01-Jan-24 2024-01-01
8   2024-01-02 2024-01-02
9     02/01/24 2024-01-02


In [19]:
invalid_dates = df_clean["date"].isna().sum()

print("Number of unparseable dates:", invalid_dates)

Number of unparseable dates: 0


In [20]:
# AI-assisted: Cleaning multiple currency formats without using regular expressions.

df_clean["amount"] = (
    df_clean["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df_clean["amount"] = pd.to_numeric(
    df_clean["amount"],
    errors="coerce"
)

print(df_clean[["Amount", "amount"]].head(10))

    Amount   amount
0    ₹2462   2462.0
1    50.00     50.0
2   ₹84728  84728.0
3    ₹1828   1828.0
4   270.00    270.0
5  Rs. 625    625.0
6  Rs. 148    148.0
7     ₹482    482.0
8  Rs. 537    537.0
9     3956   3956.0


In [21]:
invalid_amounts = df_clean["amount"].isna().sum()

print("Number of unparseable amounts:", invalid_amounts)

Number of unparseable amounts: 0


In [22]:
# AI-assisted: Standardising DR/Debit and CR/Credit into canonical values.

df_clean["type_clean"] = (
    df_clean["Type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "dr": "debit",
        "debit": "debit",
        "cr": "credit",
        "credit": "credit"
    })
)

print(df_clean["type_clean"].value_counts())

type_clean
debit     1322
credit       6
Name: count, dtype: int64


In [23]:
# AI-assisted: Removing exact duplicate transaction rows as required by the project.

rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates()

rows_after = len(df_clean)

duplicates_removed = rows_before - rows_after

print("Rows before removing duplicates:", rows_before)
print("Rows after removing duplicates:", rows_after)
print("Duplicates removed:", duplicates_removed)

Rows before removing duplicates: 1328
Rows after removing duplicates: 1310
Duplicates removed: 18


In [24]:
# AI-assisted: Removing rows where essential date or amount values could not be parsed.

df_clean = df_clean.dropna(
    subset=["date", "amount"]
)

print("Final cleaned dataset shape:", df_clean.shape)

Final cleaned dataset shape: (1310, 11)


In [25]:
# AI-assisted: Extracting month information from the cleaned datetime column.

df_clean["month"] = df_clean["date"].dt.month

df_clean["month_name"] = df_clean["date"].dt.strftime("%b")

print(df_clean[["date", "month", "month_name"]].head())

        date  month month_name
0 2024-01-01      1        Jan
1 2024-01-01      1        Jan
2 2024-01-01      1        Jan
3 2024-01-01      1        Jan
4 2024-01-01      1        Jan


In [26]:
# AI-assisted: Extracting day-of-week information for later analysis.

df_clean["day_of_week"] = df_clean["date"].dt.day_name()

print(df_clean[["date", "day_of_week"]].head())

        date day_of_week
0 2024-01-01      Monday
1 2024-01-01      Monday
2 2024-01-01      Monday
3 2024-01-01      Monday
4 2024-01-01      Monday


In [28]:
# AI-assisted: Extracting ISO week number for datetime-based analysis.

df_clean["week_number"] = (
    df_clean["date"]
    .dt.isocalendar()
    .week
)

print(df_clean[["date", "week_number"]].head())

        date  week_number
0 2024-01-01            1
1 2024-01-01            1
2 2024-01-01            1
3 2024-01-01            1
4 2024-01-01            1


In [29]:
# AI-assisted: Extracting hour from the HH:MM Time column.

df_clean["hour"] = (
    df_clean["Time"]
    .astype(str)
    .str[:2]
    .astype(int)
)

print(df_clean[["Time", "hour"]].head(10))

    Time  hour
0  03:11     3
1  05:44     5
2  09:35     9
3  14:07    14
4  14:23    14
5  14:48    14
6  14:50    14
7  21:44    21
8  05:18     5
9  06:55     6


In [30]:
print("=" * 60)
print("TRANSACTION PARSER CHECK")
print("=" * 60)

print("Final transactions:", len(df_clean))
print("Duplicate rows:", df_clean.duplicated().sum())
print("Invalid dates:", df_clean["date"].isna().sum())
print("Invalid amounts:", df_clean["amount"].isna().sum())

print("\nImportant data types:")
print(df_clean[["date", "amount", "hour"]].dtypes)

TRANSACTION PARSER CHECK
Final transactions: 1310
Duplicate rows: 0
Invalid dates: 0
Invalid amounts: 0

Important data types:
date      datetime64[ns]
amount           float64
hour               int64
dtype: object


FEATURE 2: VENDOR EXTRACTOR

In [31]:
# AI-assisted: Inspecting the actual transaction descriptions before creating vendor mappings.

unique_descriptions = (
    df_clean["Description"]
    .dropna()
    .unique()
)

print("Number of unique descriptions:", len(unique_descriptions))

for description in sorted(unique_descriptions):
    print(description)

Number of unique descriptions: 283
AIRTEL POSTPAID
AMAZON IN
AMAZON PRIME VIDEO
AMAZON SELLER SVCS
AMAZONIN MARKETPLACE
AMZN PRIME
AMZN-INTPYMT
ANI Technologies
ATM-WDL-HDFC-3609
ATM-WDL-HDFC-4942
ATM-WDL-HDFC-8030
ATM-WDL-HDFC-8253
ATM-WDL-HDFC-9140
ATM-WDL-ICICI-3918
ATM-WDL-ICICI-4172
ATM-WDL-ICICI-4739
ATM-WDL-ICICI-5025
ATM-WDL-ICICI-6478
ATM-WDL-ICICI-9135
ATM-WDL-SBI-0237
ATM-WDL-SBI-0279
ATM-WDL-SBI-0874
ATM-WDL-SBI-4080
ATM-WDL-SBI-4084
ATM-WDL-SBI-5715
AVENUE SUPERMARTS
Amazon Pay India
BANGALORE ELEC SUPPLY
BESCOM ELEC BILL
BHARTI AIRTEL LTD
BHIM SWIGGY
BHIM ZEPTO
BHIM-BLINKIT
BHIM-BMTC
BIGBASKET BANGALORE
BIGTREE ENTERTAINMENT
BLINKIT BANGALORE
BMS MOVIE TICKETS
BMTC BUS PASS
BUNDL TECH-INSTAMART
BUNDL Tech P L
BWSSB WATER BILL
COFFEE DAY GLOBAL
DISNEY HOTSTAR
FKART INTRNET
FLIPKART INDIA
FSN E-COMMERCE
Flipkart Internet
GROFERS INDIA P L
GROWW INVEST TECH
IMPS ZERODHA-COIN
IMPS-RENT-LANDLORD-35126704
IMPS-RENT-LANDLORD-36852906
IMPS-RENT-LANDLORD-39598076
IMPS-RENT-LANDLOR

In [34]:
# AI-assisted: Vendor keyword dictionary created from inspection of the provided dataset.
# No regular expressions are used.

vendor_patterns = {

    # Special transaction types
    "Cash Withdrawal": [
        "ATM-WDL"
    ],

    "P2P Transfer": [
        "UPI-AMAN-",
        "UPI-ANKIT-",
        "UPI-KARAN-",
        "UPI-NEHA-",
        "UPI-PRIYA-",
        "UPI-SNEHA-",
        "UPI-VIKAS-",
        "IMPS-RENT-LANDLORD"
    ],

    "Salary": [
        "TECHCRUSH LABS-SALARY"
    ],

    # Swiggy Instamart must appear before Swiggy
    "Instamart": [
        "SWIGGY-INSTAMART",
        "BUNDL TECH-INSTAMART",
        "INSTAMART"
    ],

    "Swiggy": [
        "SWIGGY",
        "BUNDL"
    ],

    "Zomato": [
        "ZOMATO"
    ],

    # Amazon Prime must appear before Amazon
    "Amazon Prime": [
        "AMAZON PRIME",
        "AMZN PRIME",
        "UPI-AMAZON-PRIME"
    ],

    "Amazon": [
        "AMAZON",
        "AMZN-INTPYMT",
        "AMAZONPAY"
    ],

    "Blinkit": [
        "BLINKIT"
    ],

    "Zepto": [
        "ZEPTO"
    ],

    "BigBasket": [
        "BIGBASKET",
        "INNOVATIVE RETAIL"
    ],

    "Grofers": [
        "GROFERS"
    ],

    "DMart": [
        "DMART",
        "AVENUE SUPERMARTS"
    ],

    "KiranaKart": [
        "KIRANAKART"
    ],

    "Flipkart": [
        "FLIPKART",
        "FKART"
    ],

    "Myntra": [
        "MYNTRA"
    ],

    "Nykaa": [
        "NYKAA",
        "FSN E-COMMERCE"
    ],

    "Zerodha": [
        "ZERODHA"
    ],

    "Groww": [
        "GROWW",
        "NEXTBILLION-GROWW"
    ],

    "Uber": [
        "UBER"
    ],

    "Ola": [
        "OLA ELECTRIC",
        "OLA",
        "ANI TECHNOLOGIES"
    ],

    "Rapido": [
        "RAPIDO"
    ],

    "BMTC": [
        "BMTC",
        "TUMMOC-BMTC"
    ],

    "Roppen": [
        "ROPPEN"
    ],

    "Airtel": [
        "AIRTEL",
        "BHARTI AIRTEL"
    ],

    "Jio": [
        "JIO",
        "RELIANCE JIO"
    ],

    "Vi": [
        "VODAFONE IDEA",
        "VI POSTPAID",
        "UPI-VI-RECHARGE"
    ],

    "BESCOM": [
        "BESCOM",
        "BANGALORE ELEC SUPPLY"
    ],

    "BWSSB": [
        "BWSSB"
    ],

    "Netflix": [
        "NETFLIX"
    ],

    "Disney+ Hotstar": [
        "HOTSTAR",
        "DISNEY HOTSTAR"
    ],

    "Spotify": [
        "SPOTIFY"
    ],

    "BookMyShow": [
        "BOOKMYSHOW",
        "BMS MOVIE TICKETS",
        "BIGTREE ENTERTAINMENT"
    ],

    "Star India": [
        "STAR INDIA"
    ],

    "Cafe Coffee Day": [
        "COFFEE DAY",
        "UPI-CCD"
    ],

    "Starbucks": [
        "STARBUCKS"
    ],

    "Third Wave Coffee": [
        "THIRD WAVE",
        "THIRDWAVE",
        "TWC INDIA"
    ],

    "Restaurants": [
        "TRUFFLES",
        "DINEOUT",
        "EMPIRE RESTAURANT",
        "MEGHANA",
        "BANGALORE RESTAURANT",
        "UPI-RESTAURANT"
    ],

    "Indian Oil": [
        "INDIAN OIL",
        "UPI-IOC"
    ],

    "HP": [
        "HP PETROL"
    ],

    "BPCL": [
        "BPCL"
    ]
}

In [35]:
# AI-assisted: Function for merchant/vendor normalisation using dictionary keywords.

def extract_vendor(description):

    text = str(description).upper()

    for vendor, keywords in vendor_patterns.items():

        for keyword in keywords:

            if keyword.upper() in text:
                return vendor

    return "Uncategorised"

In [36]:
# AI-assisted: Applying the vendor extraction function to every transaction.

df_clean["vendor_clean"] = (
    df_clean["Description"]
    .apply(extract_vendor)
)

print("Vendor extraction completed.")

Vendor extraction completed.


In [38]:
print(
    "Number of canonical vendors:",
    df_clean["vendor_clean"].nunique()
)

Number of canonical vendors: 41


In [39]:
print("Top vendors by transaction count:")
print(
    df_clean["vendor_clean"]
    .value_counts()
    .head(15)
)

Top vendors by transaction count:
vendor_clean
Swiggy               176
Zomato               121
Ola                   87
Amazon                76
Restaurants           73
Uber                  71
Instamart             67
Zepto                 58
Flipkart              47
Starbucks             42
Rapido                41
Blinkit               40
BMTC                  37
Third Wave Coffee     31
Cafe Coffee Day       26
Name: count, dtype: int64


In [40]:
# AI-assisted: Identifying descriptions that the vendor dictionary failed to classify.

uncategorised_vendor_rows = df_clean[
    df_clean["vendor_clean"] == "Uncategorised"
]

print(
    "Number of uncategorised transactions:",
    len(uncategorised_vendor_rows)
)

print("\nUncategorised descriptions:")

print(
    uncategorised_vendor_rows["Description"]
    .unique()
)

Number of uncategorised transactions: 0

Uncategorised descriptions:
[]


In [41]:
print("BUNDL transactions:")
print(
    df_clean[
        df_clean["Description"]
        .astype(str)
        .str.contains("BUNDL", case=False, regex=False)
    ][["Description", "vendor_clean"]]
    .head(10)
)

print("\nP2P transactions:")
print(
    df_clean[
        df_clean["Description"]
        .astype(str)
        .str.contains("UPI-", case=False, regex=False)
        &
        df_clean["vendor_clean"].eq("P2P Transfer")
    ][["Description", "vendor_clean"]]
    .head(10)
)

print("\nATM transactions:")
print(
    df_clean[
        df_clean["Description"]
        .astype(str)
        .str.contains("ATM-WDL", case=False, regex=False)
    ][["Description", "vendor_clean"]]
    .head(10)
)

BUNDL transactions:
              Description vendor_clean
64         BUNDL Tech P L       Swiggy
190  BUNDL TECH-INSTAMART    Instamart
236        BUNDL Tech P L       Swiggy
321        BUNDL Tech P L       Swiggy
322        BUNDL Tech P L       Swiggy
340        BUNDL Tech P L       Swiggy
364        BUNDL Tech P L       Swiggy
434        BUNDL Tech P L       Swiggy
479        BUNDL Tech P L       Swiggy
534        BUNDL Tech P L       Swiggy

P2P transactions:
               Description  vendor_clean
3     UPI-AMAN-8934@OKAXIS  P2P Transfer
27    UPI-AMAN-0816@OKAXIS  P2P Transfer
36   UPI-ANKIT-6430@OKAXIS  P2P Transfer
48   UPI-VIKAS-6060@OKAXIS  P2P Transfer
87   UPI-PRIYA-2221@OKAXIS  P2P Transfer
315   UPI-NEHA-9795@OKAXIS  P2P Transfer
418   UPI-NEHA-5906@OKAXIS  P2P Transfer
484  UPI-VIKAS-5416@OKAXIS  P2P Transfer
586  UPI-SNEHA-0942@OKAXIS  P2P Transfer
613  UPI-KARAN-5789@OKAXIS  P2P Transfer

ATM transactions:
             Description     vendor_clean
140     ATM-WDL-SBI-

FEATURE 3: CATEGORY TAGGER

In [42]:
# AI-assisted: Mapping canonical vendors to project spending categories.

category_map = {

    # Food
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    # Quick Commerce
    "Instamart": "Quick Commerce",
    "Blinkit": "Quick Commerce",
    "Zepto": "Quick Commerce",

    # E-commerce
    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",

    # Groceries
    "BigBasket": "Groceries",
    "Grofers": "Groceries",
    "DMart": "Groceries",
    "KiranaKart": "Groceries",

    # Transport
    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",
    "Roppen": "Transport",

    # Cafe
    "Cafe Coffee Day": "Cafe",
    "Starbucks": "Cafe",
    "Third Wave Coffee": "Cafe",

    # Restaurants
    "Restaurants": "Restaurants",

    # Subscriptions
    "Amazon Prime": "Subscriptions",
    "Netflix": "Subscriptions",
    "Disney+ Hotstar": "Subscriptions",
    "Spotify": "Subscriptions",

    # Utilities
    "Airtel": "Utilities",
    "Jio": "Utilities",
    "Vi": "Utilities",
    "BESCOM": "Utilities",
    "BWSSB": "Utilities",

    # Investments
    "Zerodha": "Investments",
    "Groww": "Investments",

    # Fuel
    "Indian Oil": "Fuel",
    "HP": "Fuel",
    "BPCL": "Fuel",

    # Entertainment
    "BookMyShow": "Entertainment",
    "Star India": "Entertainment",

    # Special categories
    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal",

    # Income
    "Salary": "Income"
}

In [43]:
# AI-assisted: Applying the vendor-to-category mapping.

df_clean["category"] = (
    df_clean["vendor_clean"]
    .map(category_map)
    .fillna("Uncategorised")
)

print("Category tagging completed.")

Category tagging completed.


In [44]:
print("Transaction count by category:")
print(
    df_clean["category"]
    .value_counts()
)

Transaction count by category:
category
Food Delivery        297
Transport            250
Quick Commerce       165
E-commerce           162
Cafe                  99
Restaurants           73
Groceries             69
Utilities             43
Subscriptions         38
Fuel                  28
Personal Transfer     24
Investments           23
Cash Withdrawal       17
Entertainment         16
Income                 6
Name: count, dtype: int64


In [45]:
uncategorised_categories = df_clean[
    df_clean["category"] == "Uncategorised"
]

print(
    "Uncategorised category transactions:",
    len(uncategorised_categories)
)

print(
    uncategorised_categories[
        ["Description", "vendor_clean"]
    ].head(20)
)

Uncategorised category transactions: 0
Empty DataFrame
Columns: [Description, vendor_clean]
Index: []


PREPARE DEBIT DATA

In [46]:
# AI-assisted: Creating a DataFrame containing debit transactions for spending analysis.

debit_data = df_clean[
    df_clean["type_clean"] == "debit"
].copy()

print("Debit transactions:", len(debit_data))

Debit transactions: 1304


In [47]:
# AI-assisted: Creating a DataFrame containing credit transactions for income analysis.

credit_data = df_clean[
    df_clean["type_clean"] == "credit"
].copy()

print("Credit transactions:", len(credit_data))

Credit transactions: 6


FEATURE 4: SPENDING OVERVIEW

In [48]:
# AI-assisted: Calculating the headline financial metrics.

total_credits = credit_data["amount"].sum()

total_debits = debit_data["amount"].sum()

net_change = total_credits - total_debits

if total_credits != 0:
    savings_rate = (
        net_change / total_credits
    ) * 100
else:
    savings_rate = 0

print("Total credits : ₹{:,.2f}".format(total_credits))
print("Total debits  : ₹{:,.2f}".format(total_debits))
print("Net change    : ₹{:,.2f}".format(net_change))
print("Savings rate  : {:.2f}%".format(savings_rate))

Total credits : ₹509,774.00
Total debits  : ₹1,678,901.00
Net change    : ₹-1,169,127.00
Savings rate  : -229.34%


In [49]:
# AI-assisted: Calculating category-level spending totals.

category_spend = (
    debit_data
    .groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
)

print("Top spending categories:")
print(category_spend.head(10))

Top spending categories:
category
E-commerce           593769.0
Investments          248160.0
Personal Transfer    132599.0
Food Delivery        129054.0
Restaurants          117737.0
Fuel                  89303.0
Quick Commerce        81874.0
Groceries             73200.0
Transport             57474.0
Cash Withdrawal       45500.0
Name: amount, dtype: float64


In [50]:
# AI-assisted: Calculating each category's share of total debit spending.

category_percentage = (
    category_spend / total_debits
) * 100

category_summary = pd.DataFrame({
    "amount": category_spend,
    "percentage": category_percentage
})

print(category_summary.head(10))

                     amount  percentage
category                               
E-commerce         593769.0   35.366528
Investments        248160.0   14.781098
Personal Transfer  132599.0    7.897964
Food Delivery      129054.0    7.686814
Restaurants        117737.0    7.012742
Fuel                89303.0    5.319134
Quick Commerce      81874.0    4.876643
Groceries           73200.0    4.359995
Transport           57474.0    3.423311
Cash Withdrawal     45500.0    2.710106


In [51]:
print("=" * 60)
print("TOP 5 CATEGORIES")
print("=" * 60)

for category, amount in category_spend.head(5).items():

    percentage = (
        amount / total_debits
    ) * 100

    print(
        f"{category:<20} "
        f"₹{amount:>12,.0f} "
        f"{percentage:>6.2f}%"
    )

TOP 5 CATEGORIES
E-commerce           ₹     593,769  35.37%
Investments          ₹     248,160  14.78%
Personal Transfer    ₹     132,599   7.90%
Food Delivery        ₹     129,054   7.69%
Restaurants          ₹     117,737   7.01%


In [52]:
# AI-assisted: Calculating vendor-level spending totals.

vendor_spend = (
    debit_data
    .groupby("vendor_clean")["amount"]
    .sum()
    .sort_values(ascending=False)
)

print("Top vendors by spending:")
print(vendor_spend.head(10))

Top vendors by spending:
vendor_clean
Amazon             318422.0
Zerodha            210000.0
Flipkart           177510.0
P2P Transfer       132599.0
Restaurants        117737.0
Swiggy              73738.0
Myntra              69529.0
Indian Oil          56335.0
Zomato              55316.0
Cash Withdrawal     45500.0
Name: amount, dtype: float64


In [53]:
transaction_count = len(df_clean)

unique_vendor_count = (
    df_clean["vendor_clean"]
    .nunique()
)

print("Total transactions:", transaction_count)
print("Unique vendors:", unique_vendor_count)

Total transactions: 1310
Unique vendors: 41


In [54]:
# AI-assisted: Creating the category-by-month spending matrix.

month_pivot = debit_data.pivot_table(
    values="amount",
    index="category",
    columns="month",
    aggfunc="sum",
    fill_value=0
)

print(month_pivot)

month                    1        2         3        4        5         6
category                                                                 
Cafe                3690.0   4273.0    5448.0   6564.0   5668.0    5802.0
Cash Withdrawal     2000.0   5000.0    8000.0   5500.0   8000.0   17000.0
E-commerce         97134.0  92773.0  103772.0  68098.0  95001.0  136991.0
Entertainment       1263.0   1746.0    2856.0   3366.0      0.0    1914.0
Food Delivery      20890.0  21452.0   20850.0  23054.0  22167.0   20641.0
Fuel               30322.0   2079.0   26164.0  18718.0   9138.0    2882.0
Groceries          20593.0   9849.0    6971.0  13773.0  13546.0    8468.0
Investments        38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Personal Transfer  25852.0  22285.0   22275.0  18663.0  21412.0   22112.0
Quick Commerce      9853.0  16187.0   17297.0  14547.0  11360.0   12630.0
Restaurants        16320.0  21772.0   28313.0   7711.0  22286.0   21335.0
Subscriptions       4256.0   4594.0   

In [55]:
month_names = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun"
}

month_pivot = month_pivot.rename(
    columns=month_names
)

print(month_pivot)

month                  Jan      Feb       Mar      Apr      May       Jun
category                                                                 
Cafe                3690.0   4273.0    5448.0   6564.0   5668.0    5802.0
Cash Withdrawal     2000.0   5000.0    8000.0   5500.0   8000.0   17000.0
E-commerce         97134.0  92773.0  103772.0  68098.0  95001.0  136991.0
Entertainment       1263.0   1746.0    2856.0   3366.0      0.0    1914.0
Food Delivery      20890.0  21452.0   20850.0  23054.0  22167.0   20641.0
Fuel               30322.0   2079.0   26164.0  18718.0   9138.0    2882.0
Groceries          20593.0   9849.0    6971.0  13773.0  13546.0    8468.0
Investments        38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Personal Transfer  25852.0  22285.0   22275.0  18663.0  21412.0   22112.0
Quick Commerce      9853.0  16187.0   17297.0  14547.0  11360.0   12630.0
Restaurants        16320.0  21772.0   28313.0   7711.0  22286.0   21335.0
Subscriptions       4256.0   4594.0   

In [56]:
# AI-assisted: Calculating total debit spending for each month.

monthly_total = (
    debit_data
    .groupby("month")["amount"]
    .sum()
    .rename(index=month_names)
)

print("Monthly spending:")
print(monthly_total)

Monthly spending:
month
Jan    290767.0
Feb    234071.0
Mar    330458.0
Apr    251382.0
May    277306.0
Jun    294917.0
Name: amount, dtype: float64


In [57]:
print("=" * 60)
print("MONTHLY SPENDING TREND")
print("=" * 60)

maximum_monthly_spend = monthly_total.max()

for month, amount in monthly_total.items():

    if maximum_monthly_spend > 0:
        bar_length = int(
            (amount / maximum_monthly_spend) * 30
        )
    else:
        bar_length = 0

    bar = "#" * bar_length

    print(
        f"{month:<5} "
        f"₹{amount:>12,.0f} "
        f"{bar}"
    )

MONTHLY SPENDING TREND
Jan   ₹     290,767 ##########################
Feb   ₹     234,071 #####################
Mar   ₹     330,458 ##############################
Apr   ₹     251,382 ######################
May   ₹     277,306 #########################
Jun   ₹     294,917 ##########################


In [58]:
# AI-assisted: Calculating January-to-June percentage change by category.

category_growth = {}

for category in month_pivot.index:

    january_value = month_pivot.loc[category, "Jan"]
    june_value = month_pivot.loc[category, "Jun"]

    if january_value != 0:

        growth = (
            (june_value - january_value)
            / january_value
        ) * 100

        category_growth[category] = growth

growth_series = pd.Series(
    category_growth
).sort_values(
    ascending=False
)

print("Biggest category growth:")
print(growth_series.head())

print("\nBiggest category decline:")
print(growth_series.tail())

Biggest category growth:
Cash Withdrawal    750.000000
Cafe                57.235772
Entertainment       51.543943
E-commerce          41.033006
Restaurants         30.729167
dtype: float64

Biggest category decline:
Personal Transfer   -14.466966
Transport           -36.428896
Investments         -39.295379
Groceries           -58.879231
Fuel                -90.495350
dtype: float64


In [59]:
# AI-assisted: Creating a category-by-hour spending matrix.

time_matrix = debit_data.pivot_table(
    values="amount",
    index="category",
    columns="hour",
    aggfunc="sum",
    fill_value=0
)

print(time_matrix)

hour                    0        1        2        3        4        5   \
category                                                                  
Cafe                 872.0    549.0      0.0    316.0      0.0      0.0   
Cash Withdrawal        0.0      0.0      0.0      0.0      0.0      0.0   
E-commerce         18650.0   9801.0  10246.0  13554.0   9738.0  13305.0   
Entertainment        562.0    620.0      0.0   1710.0   1142.0      0.0   
Food Delivery        438.0   2598.0    928.0   1702.0   2133.0   3488.0   
Fuel                2675.0   2048.0      0.0    676.0   2010.0   2105.0   
Groceries           3895.0   1536.0   2352.0  10064.0   1169.0      0.0   
Investments         4883.0  15000.0      0.0      0.0  18834.0   4496.0   
Personal Transfer      0.0      0.0      0.0      0.0      0.0      0.0   
Quick Commerce      1936.0   1821.0   1143.0   1165.0   1253.0   1448.0   
Restaurants         1411.0    960.0   1336.0   1714.0      0.0   2126.0   
Subscriptions          0.

In [60]:
food_delivery = debit_data[
    debit_data["category"] == "Food Delivery"
].copy()

print(
    "Food Delivery transactions:",
    len(food_delivery)
)

Food Delivery transactions: 297


In [61]:
# AI-assisted: Identifying Food Delivery transactions occurring between 21:00 and 02:00.

late_night_food = food_delivery[
    (food_delivery["hour"] >= 21)
    |
    (food_delivery["hour"] < 2)
]

late_night_food_percentage = (
    len(late_night_food)
    / len(food_delivery)
) * 100

print(
    "Late-night Food Delivery:",
    round(late_night_food_percentage, 2),
    "%"
)

Late-night Food Delivery: 20.54 %


In [63]:
food_hour_counts = (
    food_delivery
    .groupby("hour")
    .size()
    .sort_values(ascending=False)
)

print("Food Delivery transactions by hour:")
print(food_hour_counts)

Food Delivery transactions by hour:
hour
20    36
19    34
18    27
21    22
22    22
12    20
11    18
15    15
13    12
17    10
9     10
14     9
16     9
23     9
8      8
1      7
5      7
4      6
10     6
3      4
2      2
7      2
6      1
0      1
dtype: int64


In [64]:
print("=" * 60)
print("FOOD DELIVERY TIME-OF-DAY PATTERN")
print("=" * 60)

for hour in range(24):

    count = len(
        food_delivery[
            food_delivery["hour"] == hour
        ]
    )

    bar = "#" * min(count // 2, 30)

    print(
        f"{hour:02d}:00 | {bar} {count}"
    )

FOOD DELIVERY TIME-OF-DAY PATTERN
00:00 |  1
01:00 | ### 7
02:00 | # 2
03:00 | ## 4
04:00 | ### 6
05:00 | ### 7
06:00 |  1
07:00 | # 2
08:00 | #### 8
09:00 | ##### 10
10:00 | ### 6
11:00 | ######### 18
12:00 | ########## 20
13:00 | ###### 12
14:00 | #### 9
15:00 | ####### 15
16:00 | #### 9
17:00 | ##### 10
18:00 | ############# 27
19:00 | ################# 34
20:00 | ################## 36
21:00 | ########### 22
22:00 | ########### 22
23:00 | #### 9


In [65]:
#  Calculating the mean transaction amount within each category.

debit_data["category_mean"] = (
    debit_data
    .groupby("category")["amount"]
    .transform("mean")
)

In [66]:
#Calculating the standard deviation within each category.

debit_data["category_std"] = (
    debit_data
    .groupby("category")["amount"]
    .transform("std")
)

In [67]:
# Manual z-score calculation as required by the project.
# No scipy or other statistical library is used.

debit_data["z_score"] = (
    debit_data["amount"]
    - debit_data["category_mean"]
) / debit_data["category_std"]

In [68]:
# AI-assisted: Flagging transactions whose category-level z-score exceeds 2.

anomalies = debit_data[
    debit_data["z_score"] > 2
].copy()

anomalies = anomalies.sort_values(
    "z_score",
    ascending=False
)

print(
    "Number of anomalous transactions:",
    len(anomalies)
)

Number of anomalous transactions: 26


In [69]:
print("=" * 70)
print("TOP ANOMALOUS TRANSACTIONS")
print("=" * 70)

print(
    anomalies[
        [
            "date",
            "vendor_clean",
            "category",
            "amount",
            "z_score"
        ]
    ].head(10).to_string(index=False)
)

TOP ANOMALOUS TRANSACTIONS
      date vendor_clean    category  amount  z_score
2024-06-26       Amazon  E-commerce 22008.0 3.974482
2024-02-07       Amazon  E-commerce 21986.0 3.969715
2024-02-26  Restaurants Restaurants  8383.0 3.884639
2024-06-22  Restaurants Restaurants  7935.0 3.627582
2024-03-31  Restaurants Restaurants  7931.0 3.625287
2024-03-05       Amazon  E-commerce 19917.0 3.521407
2024-03-04  Restaurants Restaurants  7441.0 3.344131
2024-01-07  Restaurants Restaurants  7314.0 3.271260
2024-03-02       Amazon  E-commerce 18273.0 3.165188
2024-05-27     Flipkart  E-commerce 17831.0 3.069416


In [70]:
print("=" * 70)
print("TOP 5 ANOMALIES")
print("=" * 70)

for _, row in anomalies.head(5).iterrows():

    print(
        f"{row['date'].strftime('%d %b'):<10}"
        f"{row['vendor_clean']:<22}"
        f"₹{row['amount']:>10,.0f}"
        f"  z={row['z_score']:.2f}"
    )

TOP 5 ANOMALIES
26 Jun    Amazon                ₹    22,008  z=3.97
07 Feb    Amazon                ₹    21,986  z=3.97
26 Feb    Restaurants           ₹     8,383  z=3.88
22 Jun    Restaurants           ₹     7,935  z=3.63
31 Mar    Restaurants           ₹     7,931  z=3.63


FEATURE 8: SPENDING ARCHETYPE DETECTION

In [71]:
# AI-assisted: Preparing category percentages for archetype detection.

category_percentages = (
    category_spend / total_debits
) * 100

print(category_percentages)

category
E-commerce           35.366528
Investments          14.781098
Personal Transfer     7.897964
Food Delivery         7.686814
Restaurants           7.012742
Fuel                  5.319134
Quick Commerce        4.876643
Groceries             4.359995
Transport             3.423311
Cash Withdrawal       2.710106
Utilities             2.496514
Cafe                  1.872951
Subscriptions         1.532371
Entertainment         0.663827
Name: amount, dtype: float64


In [72]:
# AI-assisted: Archetype rule implemented according to the project specification.

def detect_foodie(category_percentages):

    food_categories = [
        "Food Delivery",
        "Restaurants",
        "Cafe"
    ]

    food_percentage = sum(
        category_percentages.get(
            category,
            0
        )
        for category in food_categories
    )

    if food_percentage > 25:
        return True, food_percentage

    return False, food_percentage

In [73]:
# AI-assisted: Archetype rule implemented according to the project specification.

def detect_quick_commerce(category_percentages):

    value = category_percentages.get(
        "Quick Commerce",
        0
    )

    if value > 15:
        return True, value

    return False, value

In [74]:
# AI-assisted: Archetype rule implemented according to the project specification.

def detect_shopaholic(category_percentages):

    value = category_percentages.get(
        "E-commerce",
        0
    )

    if value > 15:
        return True, value

    return False, value

In [75]:
# AI-assisted: Archetype rule implemented according to the project specification.

def detect_investor(category_percentages):

    value = category_percentages.get(
        "Investments",
        0
    )

    if value > 15:
        return True, value

    return False, value

In [76]:
# AI-assisted: Archetype rule implemented according to the project specification.

def detect_late_night_snacker(
    late_night_percentage
):

    if late_night_percentage > 50:
        return True, late_night_percentage

    return False, late_night_percentage

In [77]:
# AI-assisted: Archetype rule implemented according to the project specification.

def detect_cab_commuter(category_percentages):

    value = category_percentages.get(
        "Transport",
        0
    )

    if value > 10:
        return True, value

    return False, value

In [78]:
# AI-assisted: Archetype rule implemented according to the project specification.

def detect_subscription_lover(data):

    subscription_data = data[
        data["category"] == "Subscriptions"
    ]

    vendor_count = (
        subscription_data["vendor_clean"]
        .nunique()
    )

    if vendor_count >= 5:
        return True, vendor_count

    return False, vendor_count

In [79]:
# AI-assisted: Archetype rule implemented according to the project specification.

def detect_yolo_spender(savings_rate):

    if savings_rate < 10:
        return True, savings_rate

    return False, savings_rate

In [80]:
# AI-assisted: Archetype rule implemented according to the project specification.

def detect_disciplined_saver(savings_rate):

    if savings_rate > 40:
        return True, savings_rate

    return False, savings_rate

In [81]:
# AI-assisted: Applying all archetype detection functions to the transaction data.

archetypes = []

foodie_flag, foodie_metric = detect_foodie(
    category_percentages
)

if foodie_flag:
    archetypes.append(
        ("THE FOODIE", foodie_metric)
    )


quick_flag, quick_metric = detect_quick_commerce(
    category_percentages
)

if quick_flag:
    archetypes.append(
        ("THE QUICK COMMERCE JUNKIE", quick_metric)
    )


shop_flag, shop_metric = detect_shopaholic(
    category_percentages
)

if shop_flag:
    archetypes.append(
        ("THE SHOPAHOLIC", shop_metric)
    )


investor_flag, investor_metric = detect_investor(
    category_percentages
)

if investor_flag:
    archetypes.append(
        ("THE INVESTOR", investor_metric)
    )


late_flag, late_metric = detect_late_night_snacker(
    late_night_food_percentage
)

if late_flag:
    archetypes.append(
        ("THE LATE-NIGHT SNACKER", late_metric)
    )


cab_flag, cab_metric = detect_cab_commuter(
    category_percentages
)

if cab_flag:
    archetypes.append(
        ("THE CAB COMMUTER", cab_metric)
    )


subscription_flag, subscription_metric = detect_subscription_lover(
    debit_data
)

if subscription_flag:
    archetypes.append(
        ("THE SUBSCRIPTION LOVER", subscription_metric)
    )


yolo_flag, yolo_metric = detect_yolo_spender(
    savings_rate
)

if yolo_flag:
    archetypes.append(
        ("THE YOLO SPENDER", yolo_metric)
    )


saver_flag, saver_metric = detect_disciplined_saver(
    savings_rate
)

if saver_flag:
    archetypes.append(
        ("THE DISCIPLINED SAVER", saver_metric)
    )

In [82]:
print("=" * 70)
print("RAHUL'S SPENDING ARCHETYPES")
print("=" * 70)

for archetype, metric in archetypes:

    print(
        f"-> {archetype:<32} Metric: {metric:.2f}"
    )

RAHUL'S SPENDING ARCHETYPES
-> THE SHOPAHOLIC                   Metric: 35.37
-> THE YOLO SPENDER                 Metric: -229.34


BONUS ARCHETYPE DESCRIPTION

# Bonus Feature — Custom Archetype

## THE BENGALURU COFFEE HUNTER

A user is classified as a Bengaluru Coffee Hunter when:

- They make at least 10 Cafe transactions, and
- They spend at at least 3 distinct Cafe vendors.

This archetype is designed around Bengaluru's strong cafe culture.

In [83]:
# AI-assisted: Custom archetype implementation.

cafe_data = debit_data[
    debit_data["category"] == "Cafe"
]

cafe_transaction_count = len(cafe_data)

cafe_vendor_count = (
    cafe_data["vendor_clean"]
    .nunique()
)

coffee_hunter = (
    cafe_transaction_count >= 10
    and cafe_vendor_count >= 3
)

print(
    "Cafe transactions:",
    cafe_transaction_count
)

print(
    "Distinct cafe vendors:",
    cafe_vendor_count
)

if coffee_hunter:
    print("-> THE BENGALURU COFFEE HUNTER")
else:
    print("-> Custom archetype not detected")

Cafe transactions: 99
Distinct cafe vendors: 3
-> THE BENGALURU COFFEE HUNTER


BONUS: DAY-OF-WEEK ANALYSIS

In [84]:
# AI-assisted: Calculating spending by day of week.

day_spending = (
    debit_data
    .groupby("day_of_week")["amount"]
    .sum()
)

day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

day_spending = day_spending.reindex(
    day_order
)

print(day_spending)

day_of_week
Monday       254099.0
Tuesday      265529.0
Wednesday    307364.0
Thursday     187390.0
Friday       188350.0
Saturday     259715.0
Sunday       216454.0
Name: amount, dtype: float64


WEEKDAY VS WEEKEND

In [85]:
# AI-assisted: Comparing weekday and weekend spending.

weekdays = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday"
]

weekends = [
    "Saturday",
    "Sunday"
]

weekday_spending = day_spending[
    weekdays
].sum()

weekend_spending = day_spending[
    weekends
].sum()

print(
    "Weekday spending : ₹{:,.2f}".format(
        weekday_spending
    )
)

print(
    "Weekend spending : ₹{:,.2f}".format(
        weekend_spending
    )
)

if weekday_spending != 0:

    weekend_difference = (
        (weekend_spending - weekday_spending)
        / weekday_spending
    ) * 100

    print(
        "Weekend vs weekday difference:",
        round(weekend_difference, 2),
        "%"
    )

Weekday spending : ₹1,202,732.00
Weekend spending : ₹476,169.00
Weekend vs weekday difference: -60.41 %


BONUS: VENDOR CLEANUP AUDIT

In [87]:
# AI-assisted: Final audit of vendor extraction quality.

vendor_audit = df_clean[
    df_clean["vendor_clean"] == "Uncategorised"
][
    [
        "Description",
        "Type",
        "Amount"
    ]
]

print("=" * 60)
print("VENDOR CLEANUP AUDIT")
print("=" * 60)

if len(vendor_audit) == 0:

    print(
        "All transaction descriptions were successfully mapped."
    )

else:

    print(
        "Uncategorised transactions:",
        len(vendor_audit)
    )

    print(
        vendor_audit.to_string(index=False)
    )

VENDOR CLEANUP AUDIT
All transaction descriptions were successfully mapped.


BONUS: SPEND FORECASTING

In [89]:
# AI-assisted: Three-month average spending forecast using only Pandas and NumPy.

forecast_values = {}

for category in month_pivot.index:

    last_three_months = month_pivot.loc[
        category,
        ["Apr", "May", "Jun"]
    ].values

    forecast_values[category] = np.mean(
        last_three_months
    )

forecast_series = pd.Series(
    forecast_values
).sort_values(
    ascending=False
)

print("Forecasted next-month spending:")
print(forecast_series)

Forecasted next-month spending:
E-commerce           100030.000000
Investments           42028.000000
Food Delivery         21954.000000
Personal Transfer     20729.000000
Restaurants           17110.666667
Quick Commerce        12845.666667
Groceries             11929.000000
Fuel                  10246.000000
Cash Withdrawal       10166.666667
Transport              9807.000000
Utilities              6796.666667
Cafe                   6011.333333
Subscriptions          3121.000000
Entertainment          1760.000000
dtype: float64


ASCII BAR FUNCTION

In [90]:
# AI-assisted: Helper function for text-based report visualisation.

def create_bar(percentage, scale=1):

    bar_length = int(
        percentage * scale
    )

    return "#" * bar_length

FINAL REPORT

In [91]:
# AI-assisted: Final formatted SpendDNA report combining the calculated features.

print("=" * 75)
print("                     SpendDNA REPORT")
print("                       RAHUL SHARMA")
print("                  JANUARY - JUNE 2024")
print("=" * 75)

print()

# Executive Summary
print("EXECUTIVE SUMMARY")
print("-" * 75)

print(
    f"Total credits       : ₹{total_credits:,.0f}"
)

print(
    f"Total debits        : ₹{total_debits:,.0f}"
)

print(
    f"Net change          : ₹{net_change:,.0f}"
)

print(
    f"Savings rate        : {savings_rate:.1f}%"
)

print(
    f"Transactions        : {len(df_clean)}"
)

print(
    f"Unique vendors      : {unique_vendor_count}"
)

print()

# Top Categories
print("TOP CATEGORIES")
print("-" * 75)

for category, amount in category_spend.head(5).items():

    percentage = (
        amount / total_debits
    ) * 100

    bar = create_bar(
        percentage,
        1
    )

    print(
        f"{category:<22}"
        f"{bar:<25}"
        f"{percentage:>6.1f}% "
        f"₹{amount:>12,.0f}"
    )

print()

# Top Vendors
print("TOP VENDORS")
print("-" * 75)

for vendor, amount in vendor_spend.head(5).items():

    print(
        f"{vendor:<25}"
        f"₹{amount:>12,.0f}"
    )

print()

# Time of Day
print("TIME-OF-DAY PATTERNS")
print("-" * 75)

print(
    f"Food Delivery late-night share : "
    f"{late_night_food_percentage:.1f}%"
)

print()

# Monthly Trend
print("MONTHLY SPENDING")
print("-" * 75)

maximum_monthly_spend = monthly_total.max()

for month, amount in monthly_total.items():

    if maximum_monthly_spend > 0:

        bar_length = int(
            (amount / maximum_monthly_spend) * 30
        )

    else:

        bar_length = 0

    bar = "#" * bar_length

    print(
        f"{month:<5}"
        f"₹{amount:>12,.0f} "
        f"{bar}"
    )

print()

# Anomalies
print("TOP ANOMALIES")
print("-" * 75)

for _, row in anomalies.head(5).iterrows():

    print(
        f"{row['date'].strftime('%d %b'):<10}"
        f"{row['vendor_clean']:<22}"
        f"₹{row['amount']:>10,.0f}"
        f"  z={row['z_score']:.2f}"
    )

print()

# Archetypes
print("RAHUL'S SPENDING ARCHETYPES")
print("-" * 75)

for archetype, metric in archetypes:

    print(
        f"-> {archetype:<32}"
        f"Metric: {metric:.2f}"
    )

print()

print("=" * 75)

                     SpendDNA REPORT
                       RAHUL SHARMA
                  JANUARY - JUNE 2024

EXECUTIVE SUMMARY
---------------------------------------------------------------------------
Total credits       : ₹509,774
Total debits        : ₹1,678,901
Net change          : ₹-1,169,127
Savings rate        : -229.3%
Transactions        : 1310
Unique vendors      : 41

TOP CATEGORIES
---------------------------------------------------------------------------
E-commerce            ###################################  35.4% ₹     593,769
Investments           ##############             14.8% ₹     248,160
Personal Transfer     #######                     7.9% ₹     132,599
Food Delivery         #######                     7.7% ₹     129,054
Restaurants           #######                     7.0% ₹     117,737

TOP VENDORS
---------------------------------------------------------------------------
Amazon                   ₹     318,422
Zerodha                  ₹     210,000


# Key Insights

## 1. Category Insight

The highest spending category was **E-commerce**, accounting for approximately
**35.4%** of total debit spending.

## 2. Time-of-Day Insight

Approximately **20.5%** of Food Delivery transactions occurred during the
late-night period of 21:00 to 02:00.

## 3. Behaviour Insight

The spending analysis classified Rahul as **YOLO Spender**, primarily because
**-229.3%**.

# Reflection

This project helped me understand how real-world financial transaction data
can contain multiple date formats, currency formats, transaction type
variants, duplicate records and inconsistent merchant descriptions.

The most challenging part of the project was vendor normalisation because the
same merchant can appear under several different descriptions.

I learned how Pandas can be used to clean and transform transaction data and
how groupby, pivot tables, datetime operations and statistical calculations
can be combined to generate meaningful financial insights.

The anomaly detection feature also helped me understand how statistical
methods can be applied to identify unusual transactions within individual
spending categories.

Overall, SpendDNA helped me understand the workflow involved in transaction
analytics and the importance of data cleaning before analysis.